[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/research_assistant/openai_agents.ipynb)

# Research assistant — OpenAI Agents SDK

**Task.** Answer a user-supplied research question using typed specialist agents and deterministic safety controls.

**Read this notebook as:** configure a provider → adapt the shared model to the SDK → define planner, extractor, writer and critic agents → run the paper's eight case-study stages.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "research-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
RESEARCH_QUESTION = "Which interventions reduce household food waste, and under what conditions?"
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from agents import Agent, Runner, set_tracing_disabled
from agents.items import ModelResponse as SDKResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict, Field

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.safety import SafetyEngine, SafetyPolicy, UntrustedContent  # noqa: E402
from agentic_tutorial.schemas import CriticDecision, Message, MessageRole  # noqa: E402

catalogue = json.loads((ROOT / "data/research_assistant/evidence_catalogue.json").read_text())
fixture_path = ROOT / "data/research_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Typed agents and provider adapter

`Agent` and `Runner` perform the four model-mediated stages with typed outputs. Ordinary Python performs bounded retrieval, untrusted-content screening, deterministic claim validation, termination and reporting. Handoffs are not used here because no specialist takes over the conversation; each stage has an explicit responsibility.


In [ ]:
# --- Declarations: typed outputs, SDK adapter, agents and shared workflow helpers ---
import re


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class QueryPlan(Strict):
    schema_id: ClassVar[str] = "research.query.v1"
    queries: tuple[str, ...] = Field(min_length=1, max_length=4)


class Claim(Strict):
    source_id: str
    claim: str
    stance: Literal["supporting", "conflicting"]


class Ledger(Strict):
    schema_id: ClassVar[str] = "research.ledger.v1"
    claims: tuple[Claim, ...]


class Synthesis(Strict):
    schema_id: ClassVar[str] = "research.synthesis.v1"
    answer: str
    source_ids: tuple[str, ...]
    outcome: Literal["qualified_answer", "abstention"]


class CriticOutput(Strict):
    accepted: bool
    issues: tuple[str, ...] = ()


Claim.model_rebuild(_types_namespace={"Literal": Literal})
Ledger.model_rebuild(_types_namespace={"Claim": Claim})
Synthesis.model_rebuild(_types_namespace={"Literal": Literal})


def core_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


def sdk_input_text(value):
    """Flatten SDK input items into readable provider-neutral prompt text."""
    if isinstance(value, str):
        return value
    if isinstance(value, dict):
        if "content" in value:
            return sdk_input_text(value["content"])
        if "text" in value:
            return str(value["text"])
        return json.dumps(value, default=str, sort_keys=True)
    if isinstance(value, (list, tuple)):
        return "\n".join(part for item in value if (part := sdk_input_text(item)))
    content = getattr(value, "content", None)
    return sdk_input_text(content) if content is not None else str(value)


class FixtureModel(SDKModel):
    """Translate the repository model response into an Agents SDK model response."""

    def __init__(self):
        self.core = core_model()

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        schema = None if output_schema is None else output_schema.output_type
        request = "\n\n".join(part for part in (system_instructions, sdk_input_text(input)) if part)
        response = await self.core.generate([user(request)], response_schema=schema)
        text = json.dumps(response.structured_output, sort_keys=True)
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(annotations=[], text=text, type="output_text", logprobs=[])
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


def build_agents(model):
    return {
        "planner": Agent(
            name="Research planner",
            instructions=(
                "Create a small, sufficient set of bounded catalogue queries without "
                "assuming the answer."
            ),
            model=model,
            output_type=QueryPlan,
        ),
        "extractor": Agent(
            name="Evidence extractor",
            instructions=(
                "Extract only verbatim claims grounded in supplied records and preserve "
                "their source identifiers."
            ),
            model=model,
            output_type=Ledger,
        ),
        "writer": Agent(
            name="Evidence synthesiser",
            instructions=(
                "Answer only from validated claims, preserving conditions, conflicts and "
                "source identifiers."
            ),
            model=model,
            output_type=Synthesis,
        ),
        "critic": Agent(
            name="Evidence critic",
            instructions="Reject unsupported, unqualified or uncited synthesis.",
            model=model,
            output_type=CriticOutput,
        ),
    }


async def run_typed_agent(agent, prompt, max_turns=2):
    result = await Runner.run(agent, prompt, max_turns=max_turns)
    return result.final_output


def search(query):
    terms = set(query.casefold().split())
    return [
        record
        for record in catalogue["records"]
        if terms & set((record["title"] + " " + record["passage"]).casefold().split())
    ]


def normalise_tokens(text):
    stopwords = {
        "a",
        "an",
        "and",
        "are",
        "as",
        "at",
        "be",
        "by",
        "for",
        "from",
        "in",
        "is",
        "it",
        "of",
        "on",
        "or",
        "that",
        "the",
        "to",
        "was",
        "were",
        "with",
    }

    def canonicalise(token):
        # Lightweight singularisation is sufficient for the controlled fixture.
        if len(token) > 4 and token.endswith("s"):
            return token[:-1]
        return token

    return {
        canonicalise(token)
        for token in re.findall(r"[a-z0-9]+", text.casefold())
        if token not in stopwords
    }


def claim_grounding_score(claim, record):
    claim_tokens = normalise_tokens(claim.claim)
    passage_tokens = normalise_tokens(record["passage"])
    if len(claim_tokens) < 3:
        return 0.0
    return len(claim_tokens & passage_tokens) / len(claim_tokens)


def claim_is_grounded(claim, record, minimum_overlap=0.5):
    return claim_grounding_score(claim, record) >= minimum_overlap

In [ ]:
# --- Conceptual case-study stages ---
async def plan(agent, question):
    return await run_typed_agent(
        agent,
        f"Create at most four focused catalogue-search queries sufficient to answer "
        f"{question!r}. Do not assume particular interventions in advance.",
    )


def retrieve(query_plan):
    return {record["source_id"]: record for query in query_plan.queries for record in search(query)}


def screen_evidence(records, safety):
    assessments = {
        source_id: safety.inspect_retrieved_content(
            UntrustedContent(source_id=source_id, text=record["passage"])
        )
        for source_id, record in records.items()
    }
    safe_records = {
        source_id: record
        for source_id, record in records.items()
        if not assessments[source_id].indicators
    }
    isolated = [source_id for source_id, assessment in assessments.items() if assessment.indicators]
    return safe_records, isolated


async def extract_claims(agent, safe_records):
    return await run_typed_agent(
        agent,
        f"Extract one verbatim, source-grounded claim per record from {safe_records}. "
        "Preserve each source_id. Label reported reductions as supporting and inconsistent "
        "effects as conflicting.",
    )


def validate_claims(ledger, safe_records):
    return tuple(
        claim
        for claim in ledger.claims
        if claim.source_id in safe_records
        and claim_is_grounded(claim, safe_records[claim.source_id])
    )


async def synthesise(agent, question, claims):
    return await run_typed_agent(
        agent,
        f"Answer {question!r} using only these validated claims: {claims}. "
        "State conditions and conflicts, cite source_ids, and use outcome qualified_answer.",
    )


async def critique(agent, answer, safe_records):
    critic_output = await run_typed_agent(
        agent,
        f"Accept only if this answer is supported, appropriately qualified and cited: "
        f"{answer.model_dump()}",
    )
    critic = CriticDecision(
        accepted=critic_output.accepted,
        issues=critic_output.issues,
    )
    citations_valid = set(answer.source_ids).issubset(safe_records)
    return critic, citations_valid


def report(
    *,
    question,
    query_plan,
    retrieved,
    safe_records,
    isolated,
    claims,
    answer=None,
    critic=None,
    citations_valid=False,
    model_calls,
):
    qualified = bool(
        answer is not None
        and critic is not None
        and critic.accepted
        and citations_valid
        and answer.outcome == "qualified_answer"
    )
    outcome = "qualified_answer" if qualified else "abstention"
    if not claims:
        termination = "no_validated_claims"
    elif not citations_valid:
        termination = "invalid_citations"
    elif critic is not None and not critic.accepted:
        termination = "critic_rejected"
    else:
        termination = "criteria_met"

    return {
        "question": question,
        "query_plan": query_plan.model_dump(),
        "claims": claims,
        "answer": answer,
        "outcome": outcome,
        "termination": termination,
        "source_provenance": [] if answer is None else sorted(answer.source_ids),
        "trace": [
            {"event": "plan", "queries": query_plan.queries},
            {"event": "retrieve", "source_ids": sorted(retrieved)},
            {"event": "screen_evidence", "isolated": isolated},
            {"event": "extract_claims", "reported": len(claims)},
            {"event": "validate_claims", "count": len(claims)},
            {"event": "synthesise", "completed": answer is not None},
            {
                "event": "critique",
                "accepted": None if critic is None else critic.accepted,
                "citations_valid": citations_valid,
            },
            {"event": "report", "outcome": outcome},
        ],
        "model_calls": model_calls,
        "safe_source_ids": sorted(safe_records),
    }


async def run_research(question=RESEARCH_QUESTION):
    agents = build_agents(FixtureModel())
    safety = SafetyEngine(SafetyPolicy(allowed_tools=("catalogue_search",)))

    query_plan = await plan(agents["planner"], question)
    retrieved = retrieve(query_plan)
    safe_records, isolated = screen_evidence(retrieved, safety)
    ledger = await extract_claims(agents["extractor"], safe_records)
    claims = validate_claims(ledger, safe_records)

    if not claims:
        return report(
            question=question,
            query_plan=query_plan,
            retrieved=retrieved,
            safe_records=safe_records,
            isolated=isolated,
            claims=claims,
            model_calls=2,
        )

    answer = await synthesise(agents["writer"], question, claims)
    critic, citations_valid = await critique(agents["critic"], answer, safe_records)
    return report(
        question=question,
        query_plan=query_plan,
        retrieved=retrieved,
        safe_records=safe_records,
        isolated=isolated,
        claims=claims,
        answer=answer,
        critic=critic,
        citations_valid=citations_valid,
        model_calls=4,
    )


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_research(RESEARCH_QUESTION)
second = await run_research(RESEARCH_QUESTION) if MODEL_PROVIDER == "mock" else first
evaluation = {
    "component": len(first["claims"]) == 3,
    "trajectory": len(first["trace"]) == 8 and first["model_calls"] <= 4,
    "task": first["outcome"] == "qualified_answer",
    "safety": "fw-004" not in first["source_provenance"],
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 4},
    "fallback": "no validated claims or failed critique produces abstention",
}

## Limitation

The provider adapter validates Agents SDK orchestration with the repository's mock/local/API abstraction; it does not exercise hosted OpenAI models. The bounded catalogue and deterministic fixtures cannot establish open-domain research quality.
